# Verify the preserved P4b task-segmented schedule

Attach **exact Version 1** of `thestonedape/task-aware-eeg2text-task-segmented-schedule`, enable Internet, and enable the private Kaggle secret `GITHUB_TOKEN`. Use a CPU session. This notebook re-hashes all nine preserved files and independently decodes all 4,032,000 schedule indices. It performs no training and never accesses validation or test data.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
VERIFIER_COMMIT = '75fcd25a02e82b477623cc19bab6badbc92be412'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-segmented-schedule-preserved-verification'
DATASET_SLUG = 'thestonedape/task-aware-eeg2text-task-segmented-schedule'
PRESERVED_DATASET_VERSION = 1
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eeg2text-task-segmented-schedule-version-1'
EXPECTED_REPORT_SHA256 = '65609e0dc9ae48403a16ddf39f08403fbe0578e96ce8b01255c966252498d193'
EXPECTED_REPORT_BYTES = 2211
EXPECTED_SCHEDULE_CONTRACT_SHA256 = 'a6ea34388cd98380654f413b1440d0d5cee0b8065555b0c04a53d0db6ea12287'
EXPECTED_SCHEDULE_MANIFEST_SHA256 = '0cf2a752b5f0a67e7282bc0b4551b4792ceb3d346dd441fef6c359515885270a'
EXPECTED_SCHEDULE_REPORT_SHA256 = '5e7d953a8ca31e48d6acbd6168b146d695af449ef0ad5bc573a37d306cb7250f'
EXPECTED_RUN_METADATA_SHA256 = '8487422b900573e8b44d7e83d78eb32beb9786f8496a972c5585291c3576dc1b'
assert len(VERIFIER_COMMIT) == 40 and PRESERVED_DATASET_VERSION == 1
assert all(len(value) == 64 for value in (EXPECTED_REPORT_SHA256, EXPECTED_SCHEDULE_CONTRACT_SHA256, EXPECTED_SCHEDULE_MANIFEST_SHA256, EXPECTED_SCHEDULE_REPORT_SHA256, EXPECTED_RUN_METADATA_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    if os.path.exists(askpass):
        os.remove(askpass)
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', VERIFIER_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == VERIFIER_COMMIT
subprocess.run([sys.executable, '-m', 'unittest', 'evaluation.test_verify_task_segmented_training_schedule_output', 'evaluation.test_verify_task_segmented_training_schedule_artifact'], check=True, cwd=WORKTREE)
print({'python': platform.python_version(), 'verifier_commit': actual_commit, 'regressions': 'PASS'})

In [ ]:
EXPECTED_ARTIFACT_FILES = {
    'trial_catalog.csv', 'schedule_indices.u32le', 'schedule_units.csv',
    'schedule_audit.csv', 'task_segmented_training_schedule_manifest.json',
    'task_segmented_training_schedule_report.json',
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json', 'schedule_freeze_run_metadata.json',
}
metadata_candidates = glob.glob('/kaggle/input/**/schedule_freeze_run_metadata.json', recursive=True)
artifact_roots = []
for path in metadata_candidates:
    root = os.path.dirname(path)
    try:
        names = set(os.listdir(root))
    except OSError:
        continue
    if names == EXPECTED_ARTIFACT_FILES and all(os.path.isfile(os.path.join(root, name)) and not os.path.islink(os.path.join(root, name)) for name in names):
        artifact_roots.append(root)
artifact_roots = sorted(set(artifact_roots))
assert len(artifact_roots) == 1, ('Attach exact Version 1 of the one complete task-segmented schedule dataset', artifact_roots, metadata_candidates)
ARTIFACT_ROOT = artifact_roots[0]
print({'dataset_slug': DATASET_SLUG, 'dataset_version': PRESERVED_DATASET_VERSION, 'artifact_root': ARTIFACT_ROOT})

In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
os.makedirs(OUTPUT)
report_path = os.path.join(OUTPUT, 'schedule_artifact_verification_report.json')
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'verify_task_segmented_training_schedule_artifact.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
    '--output-report', report_path,
], check=True, cwd=WORKTREE)
with open(report_path, encoding='utf-8') as handle:
    report = json.load(handle)
assert report['status'] == 'pass'
assert report['preserved_source_id'] == PRESERVED_SOURCE_ID
assert report['catalog_rows'] == 9011 and report['global_batches_per_epoch'] == 105
assert report['shape'] == [15, 40, 105, 64]
assert report['scheduled_uint32_indices_deeply_verified'] == 4032000
assert report['schedule_contract_sha256'] == EXPECTED_SCHEDULE_CONTRACT_SHA256
assert report['schedule_manifest_sha256'] == EXPECTED_SCHEDULE_MANIFEST_SHA256
assert report['schedule_report_sha256'] == EXPECTED_SCHEDULE_REPORT_SHA256
assert report['schedule_freeze_run_metadata_sha256'] == EXPECTED_RUN_METADATA_SHA256
assert report['bounded_smoke_authorized'] is True
assert report['full_training_authorized'] is False
assert report['held_out_test_accessed'] is False
assert digest(report_path) == EXPECTED_REPORT_SHA256
assert os.path.getsize(report_path) == EXPECTED_REPORT_BYTES
metadata = {
    'status': 'pass', 'schema_version': 1, 'verifier_commit': actual_commit,
    'preserved_source_id': PRESERVED_SOURCE_ID,
    'verification_report_sha256': EXPECTED_REPORT_SHA256,
    'bounded_smoke_authorized': True, 'full_training_authorized': False,
    'held_out_test_accessed': False,
}
metadata_path = os.path.join(OUTPUT, 'verification_run_metadata.json')
with open(metadata_path, 'wb') as handle:
    handle.write((json.dumps(metadata, indent=2, sort_keys=True) + '\n').encode('utf-8'))
assert set(os.listdir(OUTPUT)) == {'schedule_artifact_verification_report.json', 'verification_run_metadata.json'}
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print({
    'status': report['status'], 'shape': report['shape'],
    'scheduled_uint32_indices_deeply_verified': report['scheduled_uint32_indices_deeply_verified'],
    'verification_report_sha256': EXPECTED_REPORT_SHA256,
    'bounded_smoke_authorized': report['bounded_smoke_authorized'],
    'full_training_authorized': report['full_training_authorized'],
})
print('TASK-SEGMENTED TRAINING SCHEDULE DATASET V1 CLEAN-REMOUNT VERIFICATION: PASS')

After the terminal PASS, the exact Version-1 schedule preservation gate is closed. The next authorized action is only the frozen two-batch, three-arm real-data smoke. The 45 scientific fits remain unauthorized.